# Eff-3DPSeg (spconv) on Kaggle

Settings: **Accelerator = GPU** (T4/P100), **Internet = On**. Add your raw Pheno4D folder as a Kaggle Dataset (one sub-folder per plant, annotated files end in `_a.txt`).

Edit the two constants in the next cell.

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/<your-username>/3d-phenomics.git'   # <- your fork
RAW_DIR  = '/kaggle/input/pheno4d'                                  # <- raw Pheno4D dataset

REPO = '/kaggle/working/3d-phenomics'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', REPO_URL, REPO], check=True)
%cd {REPO}/eff3dpseg

In [ ]:
# Sparse-conv backend. spconv has no working build for CUDA 13 notebooks (its C++ build fails),
# so use the pure-PyTorch implementation in scripts/sparse_torch.py: no install, any CUDA, a bit slower.
# Later cells run `!python scripts/...` which inherits this environment variable.
import os, torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Notebook Settings -> Accelerator -> GPU, then Run -> Restart session.')
os.environ['EFF_SPARSE_BACKEND'] = 'torch'     # 'auto' tries spconv first, then falls back
!python scripts/test_sparse_torch.py

In [ ]:
!python scripts/smoke_test.py

In [ ]:
# Convert raw txt -> npz + plant-level splits. Hold out whole plants for val/test.
# Plant names: ls $RAW_DIR   (edit --val-plants / --test-plants accordingly)
DATA = '/kaggle/working/pheno4d_npz'
!python scripts/prepare_pheno4d.py --raw {RAW_DIR} --out {DATA} --val-plants Tomato06 --test-plants Tomato07

In [ ]:
# --voxel-size is in the raw point-cloud units; the original used 0.2. Check your units!
!python scripts/train.py --data {DATA} --run-name demo --max-iter 2000 --validate-step 50 --time-limit-hours 11

### Resume after a session cut
Checkpoints live in `/kaggle/working/checkpoints` (save them as a Dataset between sessions). Re-run with:
`!python scripts/train.py --data {DATA} --run-name demo --max-iter 2000 --resume auto`

In [ ]:
!python scripts/infer.py --data {DATA} --ckpt /kaggle/working/checkpoints/demo_best.pth --phase test --out-dir /kaggle/working/preds --save-ply